# SASRec Stage 3 Attention Bias Multi-Task BPI2012 Colab Train 03

This notebook tests the Stage 3 attention-bias multi-task setting on the `anchor_ml20` backbone.

Main comparison groups:
- `anchor_single_task`
- `anchor_attention_bias_single_task`
- `anchor_multi_task`
- `anchor_attention_bias_multi_task`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_ATTNBIAS_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR:', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_ATTNBIAS_OUTPUT_DIR:', MULTITASK_ATTNBIAS_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_ATTNBIAS_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_ATTNBIAS_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [8]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260601_113446
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists.


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

Stage 3 attention-bias multi-task runs:

- backbone: `anchor_ml20`
- time-aware backbone: `delta_start + 9-bucket attention bias`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- time loss weight: `1.0`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`


## Check prerequisite reference runs


In [11]:
from pathlib import Path

baseline_required_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]
multitask_baseline_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]

checks = [
    ('baseline', BASELINE_NDCG10_OUTPUT_DIR, baseline_required_runs),
    ('single-task attention bias', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR, single_task_attnbias_runs),
    ('multi-task baseline', MULTITASK_BASELINE_OUTPUT_DIR, multitask_baseline_runs),
]

print('=' * 80)
for label, output_dir, run_names in checks:
    print(label)
    base = Path(output_dir)
    for run_name in run_names:
        run_dir = base / run_name
        print(' ', run_name, 'EXISTS' if run_dir.exists() else 'MISSING')
    print('-' * 80)


baseline
  anchor_ml20_s42 EXISTS
  anchor_ml20_s2024 EXISTS
  anchor_ml20_s7 EXISTS
--------------------------------------------------------------------------------
single-task attention bias
  attnbias_dstart_ml20_b9_s42 EXISTS
  attnbias_dstart_ml20_b9_s2024 EXISTS
  attnbias_dstart_ml20_b9_s7 EXISTS
--------------------------------------------------------------------------------
multi-task baseline
  multitask_anchor_ml20_s42 EXISTS
  multitask_anchor_ml20_s2024 EXISTS
  multitask_anchor_ml20_s7 EXISTS
--------------------------------------------------------------------------------


## Check planned attention-bias multi-task runs


In [12]:
planned_attnbias_multitask_runs = [
    'multitask_attnbias_dstart_ml20_b9_s42',
    'multitask_attnbias_dstart_ml20_b9_s2024',
    'multitask_attnbias_dstart_ml20_b9_s7',
]

output_dir = Path(MULTITASK_ATTNBIAS_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 attention-bias multi-task runs')
for run_name in planned_attnbias_multitask_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 attention-bias multi-task runs
multitask_attnbias_dstart_ml20_b9_s42 OK
multitask_attnbias_dstart_ml20_b9_s2024 OK
multitask_attnbias_dstart_ml20_b9_s7 OK


## Train attention-bias multi-task runs


In [13]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_s42
epoch=1, loss=3.3679
epoch=2, loss=1.7529
epoch=3, loss=1.5693
epoch=4, loss=1.4606
epoch=5, loss=1.3877
valid [task], Top5Acc: 0.3857, Top10Acc: 0.8256, Acc: 0.0662, MacroF1: 0.0906, TimeMAE: 68587.0841, TimeRMSE: 275932.5651, TimeMedAE: 3655.1631
valid [full], NDCG@5: 0.6669, HR@5: 0.8179, NDCG@10: 0.7275, HR@10: 0.9933, MRR: 0.6456
valid [sampled], NDCG@5: 0.5266, HR@5: 0.5274, NDCG@10: 0.5326, HR@10: 0.5467, MRR: 0.5464
test [task], Top5Acc: 0.2952, Top10Acc: 0.6795, Acc: 0.0315, MacroF1: 0.0237, T

In [14]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_s2024
epoch=1, loss=3.2477
epoch=2, loss=1.8038
epoch=3, loss=1.5880
epoch=4, loss=1.4916
epoch=5, loss=1.4214
valid [task], Top5Acc: 0.4146, Top10Acc: 0.6881, Acc: 0.0849, MacroF1: 0.1398, TimeMAE: 76839.6347, TimeRMSE: 296077.5551, TimeMedAE: 1850.6166
valid [full], NDCG@5: 0.5829, HR@5: 0.6495, NDCG@10: 0.6758, HR@10: 0.9407, MRR: 0.6042
valid [sampled], NDCG@5: 0.5189, HR@5: 0.5196, NDCG@10: 0.5234, HR@10: 0.5336, MRR: 0.5333
test [task], Top5Acc: 0.2828, Top10Acc: 0.5018, Acc: 0.0324, MacroF1: 0.0268,

In [15]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml20_b9_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml20_b9_s7
epoch=1, loss=3.3921
epoch=2, loss=1.8234
epoch=3, loss=1.5763
epoch=4, loss=1.4546
epoch=5, loss=1.3791
valid [task], Top5Acc: 0.3029, Top10Acc: 0.7309, Acc: 0.0653, MacroF1: 0.0842, TimeMAE: 70774.8850, TimeRMSE: 289241.9922, TimeMedAE: 1443.4507
valid [full], NDCG@5: 0.5943, HR@5: 0.6848, NDCG@10: 0.6833, HR@10: 0.9576, MRR: 0.6057
valid [sampled], NDCG@5: 0.5076, HR@5: 0.5086, NDCG@10: 0.5142, HR@10: 0.5290, MRR: 0.5242
test [task], Top5Acc: 0.3597, Top10Acc: 0.5850, Acc: 0.0324, MacroF1: 0.0222, Ti

In [16]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_delta_column': config.get('time_delta_column'),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [17]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [18]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]
multitask_baseline_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]
multitask_attnbias_runs = [
    'multitask_attnbias_dstart_ml20_b9_s42',
    'multitask_attnbias_dstart_ml20_b9_s2024',
    'multitask_attnbias_dstart_ml20_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
single_task_attnbias_df = rebuild_df(SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
multitask_baseline_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_attnbias_df = rebuild_df(MULTITASK_ATTNBIAS_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'anchor_single_task'

single_task_attnbias_subset = single_task_attnbias_df[single_task_attnbias_df['run_name'].isin(single_task_attnbias_runs)].copy()
single_task_attnbias_subset['variant'] = 'anchor_attnbias_single_task'

multitask_baseline_subset = multitask_baseline_df[multitask_baseline_df['run_name'].isin(multitask_baseline_runs)].copy()
multitask_baseline_subset['variant'] = 'anchor_multi_task'

multitask_attnbias_subset = multitask_attnbias_df[multitask_attnbias_df['run_name'].isin(multitask_attnbias_runs)].copy()
multitask_attnbias_subset['variant'] = 'anchor_attnbias_multi_task'

df_compare = pd.concat(
    [
        baseline_subset,
        single_task_attnbias_subset,
        multitask_baseline_subset,
        multitask_attnbias_subset,
    ],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

display_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric', 'best_epoch',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10',
    'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10',
    'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy',
    'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy',
    'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae',
    'best_valid_task_time_rmse',
    'best_valid_task_time_median_ae',
    'best_test_at_best_valid_task_accuracy',
    'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy',
    'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae',
    'best_test_at_best_valid_task_time_rmse',
    'best_test_at_best_valid_task_time_median_ae',
]

existing_display_cols = [c for c in display_cols if c in df_compare.columns]
df_compare[existing_display_cols]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,best_epoch,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_mrr,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_mrr,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_top5_accuracy,best_valid_task_top10_accuracy,best_valid_task_time_mae,best_valid_task_time_rmse,best_valid_task_time_median_ae,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_top5_accuracy,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_time_median_ae
0,multitask_attnbias_dstart_ml20_b9_s7,7,anchor_attnbias_multi_task,20,0.2,full_valid_ndcg@10,50,0.680319,0.834780,0.719083,0.954286,0.651337,0.666631,0.919326,0.692974,1.000000,0.590479,0.560260,0.561313,0.564768,0.575692,0.575858,0.147555,0.153198,0.185106,0.273666,0.199344,0.070673,0.069174,0.398806,0.700081,72629.537307,285304.457740,2686.401425,0.027570,0.019458,0.351487,0.618634,12344.991180,70498.991743,40.622211
1,multitask_attnbias_dstart_ml20_b9_s42,42,anchor_attnbias_multi_task,20,0.2,full_valid_ndcg@10,50,0.713810,0.851927,0.743438,0.948749,0.684053,0.712982,0.994861,0.714761,1.000000,0.617692,0.580058,0.584449,0.592719,0.624476,0.597263,0.161010,0.171220,0.199952,0.295916,0.211366,0.059364,0.059898,0.422177,0.586613,69265.443245,281265.372853,2957.257658,0.027049,0.020074,0.382337,0.630782,11861.273768,70649.783007,183.085221
2,multitask_attnbias_dstart_ml20_b9_s2024,2024,anchor_attnbias_multi_task,20,0.2,full_valid_ndcg@10,50,0.703145,0.838492,0.744063,0.968239,0.677378,0.817992,0.992456,0.820680,1.000000,0.760539,0.561608,0.566428,0.575255,0.609136,0.580869,0.180017,0.216893,0.241043,0.408191,0.226260,0.066090,0.063037,0.400865,0.783755,72413.146990,290667.785204,772.234459,0.028425,0.021196,0.272262,0.674929,11605.227529,71873.368725,78.030332
3,attnbias_dstart_ml20_b9_s7,7,anchor_attnbias_single_task,20,0.2,full_valid_ndcg@10,40,0.694816,0.812322,0.731681,0.933234,0.674079,0.833460,0.998642,0.833933,1.000000,0.776636,0.550907,0.567329,0.584594,0.673558,0.571303,0.349894,0.366486,0.408188,0.553189,0.395330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,attnbias_dstart_ml20_b9_s42,42,anchor_attnbias_single_task,20,0.2,full_valid_ndcg@10,30,0.692852,0.854887,0.731547,0.976377,0.657784,0.844950,0.958941,0.857427,1.000000,0.810092,0.535631,0.543183,0.553607,0.600217,0.557929,0.504964,0.524491,0.535711,0.621374,0.530831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,attnbias_dstart_ml20_b9_s2024,2024,anchor_attnbias_single_task,20,0.2,full_valid_ndcg@10,30,0.682637,0.839902,0.727796,0.977380,0.653203,0.800095,0.953875,0.815191,1.000000,0.753665,0.539500,0.543537,0.551441,0.581497,0.560101,0.311595,0.332384,0.369433,0.517223,0.355765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,multitask_anchor_ml20_s7,7,anchor_multi_task,20,0.2,full_valid_ndcg@10,45,0.711434,0.854410,0.737962,0.941655,0.678242,0.782387,0.937745,0.801466,1.000000,0.735914,0.559712,0.564315,0.575376,0.613704,0.579719,0.201450,0.245365,0.265319,0.444986,0.240381,0.062415,0.097186,0.403664,0.646811,71279.254416,287403.069435,872.565819,0.026391,0.023985,0.460549,0.595886,11392.935271,66680.324584,6.927000
7,multitask_anchor_ml20_s42,42,anchor_multi_task,20,0.2,full_valid_ndcg@10,30,0.706218,0.858147,0.743167,0.977823,0.671713,0.787125,0.939242,0.807542,1.000000,0.743154,0.537092,0.544557,0.553

In [19]:
summary_metric_cols = [
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10',
    'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10',
    'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy',
    'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy',
    'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae',
    'best_valid_task_time_rmse',
    'best_valid_task_time_median_ae',
    'best_test_at_best_valid_task_accuracy',
    'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy',
    'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae',
    'best_test_at_best_valid_task_time_rmse',
    'best_test_at_best_valid_task_time_median_ae',
]

summary_metric_cols = [c for c in summary_metric_cols if c in df_compare.columns]
summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_mrr           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_mrr           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_mrr           best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_top5_accuracy           best_valid_task_top10_accuracy           best_valid_task_time_mae              best_valid_task_time_rmse              best_valid_task_time_median_ae              best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_top5_accuracy           best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_time_median_ae           
                                              mean       std                 mean       std                    mean       std                  mean       std                mean       std                                mean       std                              mean       std                                 mean       std                               mean       std                             mean       std                      mean       std                    mean       std                       mean       std                     mean       std                   mean       std                                   mean       std                                 mean       std                                    mean       std                                  mean       std                                mean       std                     mean       std                     mean       std                          mean       std                           mean       std                     mean          std                      mean          std                           mean          std                                  mean       std                                  mean       std                                       mean       std                                        mean       std                                  mean          std                                   mean          std                                        mean        std
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

Interpretation guide:

- compare `anchor_single_task` vs `anchor_attnbias_single_task`
- compare `anchor_single_task` vs `anchor_multi_task`
- compare `anchor_multi_task` vs `anchor_attnbias_multi_task`
- use `best_test_at_best_valid_full_ndcg@10` as the main Stage 3 comparison metric
- use mean/std across `42`, `2024`, `7` for final interpretation
- if attention bias helps in multi-task, the main question is whether it recovers some of the ranking loss seen in the baseline multi-task setting
